# Direction-feature experiment

Add eight fixed geometry/movement features to the current ranker. Keep the same 78 training trips (exclude 33 and 54), saved trip folds, candidates, reference labels and hyperparameters. No additional model tuning or label changes.

First compare full-row CV with the existing 78-trip model. Then refit and score the same ten student-test trips. This is repeated development evaluation, not a fresh independent test.

The original features, models and export remain available. New features and results have separate paths.

In [1]:
%run 06_model_helpers.ipynb
%run 05_direction_features.ipynb
import xgboost as xgb
from pyproj import Transformer
import pyarrow as pa
import pyarrow.parquet as pq
from time import perf_counter
from datetime import datetime
from zoneinfo import ZoneInfo

feature_folder = Path("outputs/direction_features")
output = Path("results/direction_features")
feature_folder.mkdir(parents=True, exist_ok=True)
output.mkdir(parents=True, exist_ok=True)
excluded = json.loads(Path("outputs/training_exclusions.json").read_text())["excluded_trip_ids"]
base_columns = contract["feature_columns"]
feature_columns = base_columns + direction_columns
assert len(feature_columns) == len(set(feature_columns)) == 32
params = json.loads(Path("results/exclude_33_54/config.json").read_text())["params"]
n_jobs = 8
config = {"excluded_trip_ids": excluded, "base_features": base_columns, "added_features": direction_columns,
          "half_windows_seconds": [3, 5], "heading_min_speed_mps": 0.5, "heading_min_displacement_m": 1,
          "params": params, "seed": seed, "fold_file": contract["fold_file"],
          "candidate_settings": contract["candidate_settings"], "labels_modified": False}
(feature_folder / "config.json").write_text(json.dumps(config, indent=2) + "\n")
(feature_folder / "feature_columns.json").write_text(json.dumps(feature_columns, indent=2) + "\n")

740

## Build additional features, one trip at a time

Read the existing predictor tables; retain their values and labels. Read only raw nodes, arcs and GPS observations for the additional features. Reference routes and point targets are not inputs to the direction calculations.

The shared helper uses time windows, not a fixed number of rows, so large timestamp gaps are not treated as consecutive one-second observations. Raw trip boundaries and undefined directions remain missing.

In [2]:
checks = []
partitions = [("train", "outputs/features/features.parquet", "train.parquet", "train"),
              ("student_test", "outputs/student_test/features.parquet", "student_test.parquet", "student_test")]
for partition, source_path, filename, raw_partition in partitions:
    source = pq.ParquetFile(source_path)
    writer = None
    try:
        for group in range(source.num_row_groups):
            frame = source.read_row_group(group).to_pandas()
            assert frame["trip_id"].nunique() == 1
            trip_id = frame["trip_id"].iloc[0]
            if partition == "train" and trip_id in excluded:
                continue
            before = frame[["trip_id", "group_id", "point_id", "segment_id", "label"]].copy()
            folder = Path("Student folder") / raw_partition / trip_id
            nodes = pd.read_csv(folder / f"{trip_id}.nodes", sep=r"\s+", header=None).to_numpy()
            arcs = pd.read_csv(folder / f"{trip_id}.arcs", sep=r"\s+", header=None).to_numpy()
            track = pd.read_csv(folder / f"{trip_id}.track", sep=r"\s+", header=None, names=["lon", "lat", "time"])
            zone = int((track["lon"].mean() + 180) // 6) + 1
            epsg = (32600 if track["lat"].mean() >= 0 else 32700) + zone
            projection = Transformer.from_crs(4326, epsg, always_xy=True)
            nodes_xy = np.column_stack(projection.transform(nodes[:, 0], nodes[:, 1]))
            gps_xy = np.column_stack(projection.transform(track["lon"], track["lat"]))
            extra = direction_features(frame, nodes_xy, arcs, gps_xy, track["time"].to_numpy())
            assert not np.isinf(extra.to_numpy()).any()
            for seconds in [3, 5]:
                assert extra[f"direction_reliability_{seconds}s"].dropna().between(0, 1).all()
                assert extra[f"heading_diff_smoothed_{seconds}s"].dropna().between(0, 180).all()
            frame[direction_columns] = extra
            nearest = frame["distance"] - frame["distance_above_nearest_m"]
            frame["distance_ratio"] = ((frame["distance"] + 1) / (nearest + 1)).astype("float32")
            pd.testing.assert_frame_equal(frame[before.columns], before)
            if partition == "student_test":
                assert frame["label"].isna().all()
            table = pa.Table.from_pandas(frame[list(before.columns) + feature_columns], preserve_index=False)
            if writer is None:
                writer = pq.ParquetWriter(feature_folder / filename, table.schema, compression="zstd")
            writer.write_table(table, row_group_size=len(frame))
            checks.append({"partition": partition, "trip_id": trip_id, "rows": len(frame)})
            print("Features:", partition, trip_id, flush=True)
    finally:
        if writer is not None:
            writer.close()
summary = pd.DataFrame(checks)
summary.to_csv(feature_folder / "summary.csv", index=False)
assert summary[summary["partition"] == "train"]["trip_id"].nunique() == 78
assert summary[summary["partition"] == "student_test"]["trip_id"].nunique() == 10
print("Built", len(feature_columns), "predictors; labels and candidates unchanged.", flush=True)

Features: train 00000001


Features: train 00000002


Features: train 00000003


Features: train 00000004


Features: train 00000005


Features: train 00000006


... (79 outputs left out)

Features: student_test 00000076


Features: student_test 00000077


Features: student_test 00000091


Built 32 predictors; labels and candidates unchanged.


## Load the same CV population

Keep each surviving trip in its original fold. All valid candidate rows remain present; training still excludes singleton/no-positive ranker groups. CV point and candidate counts must match the previous exclusion experiment exactly.

In [3]:
data = pd.read_parquet(feature_folder / "train.parquet")
data = data[data["label"].notna()].reset_index(drop=True)
data["label"] = data["label"].astype("int8")
folds = pd.read_csv(contract["fold_file"], dtype={"trip_id": str}).set_index("group_id")["fold"]
data["fold"] = data["group_id"].map(folds)
points = pd.read_csv("outputs/candidates/points.csv", dtype={"trip_id": str})
points = points[~points["bad_gps"] & ~points["trip_id"].isin(excluded)].copy()
points["fold"] = points["group_id"].map(folds)
previous_config = json.loads(Path("results/exclude_33_54/config.json").read_text())
assert len(data) == previous_config["candidate_rows"] and len(points) == previous_config["training_points"]
assert data["trip_id"].nunique() == 78 and data["group_id"].is_monotonic_increasing
assert data["fold"].notna().all() and data.groupby("trip_id")["fold"].nunique().eq(1).all()

In [4]:
def fit_ranker(frame, params):
    sizes = frame.groupby("group_id")["label"].agg(["size", "sum"])
    eligible = sizes.index[(sizes["size"] >= 2) & (sizes["sum"] == 1)]
    training = frame[frame["group_id"].isin(eligible)]
    assert training["group_id"].is_monotonic_increasing
    groups = training.groupby("group_id", sort=False).size().to_numpy()
    model = xgb.XGBRanker(objective="rank:ndcg", eval_metric="ndcg@1", tree_method="hist",
                          random_state=seed, n_jobs=n_jobs, **params)
    model.fit(training[feature_columns], training["label"], group=groups, verbose=False)
    return model


## Fixed-setting five-fold CV

Save progress after each fold. Metrics use the same point denominator, including candidate misses. Also save the per-point winning arc so direction errors can be examined later.

In [5]:
records = []
winner_parts = []
oof_probability = np.full(len(data), np.nan, dtype=np.float64)
for fold in range(5):
    training = data[data["fold"] != fold]
    validation = data[data["fold"] == fold]
    valid_points = points[points["fold"] == fold]
    assert set(training["trip_id"]).isdisjoint(validation["trip_id"])
    start = perf_counter()
    model = fit_ranker(training, params)
    fit_s = perf_counter() - start
    start = perf_counter()
    probability = point_softmax(model.predict(validation[feature_columns]), validation["group_id"])
    predict_s = perf_counter() - start
    oof_probability[validation.index] = probability
    records.append({"fold": fold, **score_points(validation, probability, valid_points), "fit_s": fit_s, "predict_s": predict_s})
    cv = pd.DataFrame(records)
    cv.to_csv(output / "cv_metrics.csv", index=False)
    choices = validation[["group_id", "segment_id"]].copy()
    choices["score"] = probability
    choices = choices.sort_values(["group_id", "segment_id"]).reset_index(drop=True)
    winner_parts.append(choices.loc[choices.groupby("group_id")["score"].idxmax(), ["group_id", "segment_id"]])
    print("Completed fold", fold, "| Top-1", records[-1]["top1"], flush=True)

assert np.isfinite(oof_probability).all()
pooled = score_points(data, oof_probability, points)
(output / "pooled_metrics.json").write_text(json.dumps(pooled, indent=2) + "\n")
oof = data[["group_id", "segment_id", "fold"]].copy()
oof["score"] = oof_probability
oof.to_csv(output / "oof_xgb_ranker.csv", index=False)
pd.concat(winner_parts).sort_values("group_id").to_csv(output / "oof_winners.csv", index=False)
old_cv = pd.read_csv("results/exclude_33_54/cv_metrics.csv")
cv_comparison = old_cv[["fold", "top1", "macro_f1", "log_loss"]].merge(
    cv[["fold", "top1", "macro_f1", "log_loss"]], on="fold", suffixes=("_before", "_direction"))
cv_comparison.to_csv(output / "cv_comparison.csv", index=False)
print("New pooled CV:", pooled, flush=True)
display(cv_comparison)

Completed fold 0 | Top-1 0.8685923152358028


Completed fold 1 | Top-1 0.8632583111622444


Completed fold 2 | Top-1 0.9364832163308335


Completed fold 3 | Top-1 0.9384455317473708


Completed fold 4 | Top-1 0.9137223476846118


New pooled CV: {'top1': 0.9066663016398828, 'macro_f1': 0.9524783057030906, 'precision': 0.9066663016398828, 'recall': 0.907018908437589, 'log_loss': 0.2856609945683332, 'candidate_coverage': 0.9996112464752102, 'points': 182635, 'candidate_misses': 71, 'confusion_matrix': [[9002537, 17046], [16975, 165589]]}


,fold,top1_before,macro_f1_before,log_loss_before,top1_direction,macro_f1_direction,log_loss_direction
0,0,0.854239,0.926013,0.442841,0.868592,0.933298,0.397848
1,1,0.857218,0.927134,0.406284,0.863258,0.930216,0.375662
2,2,0.922587,0.960259,0.234625,0.936483,0.967393,0.185120
3,3,0.934649,0.966680,0.217018,0.938446,0.968615,0.205596
4,4,0.900172,0.949529,0.327480,0.913722,0.956439,0.304438


## Final refit and repeated student-test comparison

Use the same fixed parameters and training population. Do not choose features or hyperparameters from these student-test results. Prediction selects only the 32 predictor columns; attach reference labels afterwards for evaluation.

In [6]:
start = perf_counter()
model = fit_ranker(data, params)
final_fit_s = perf_counter() - start
model.save_model(output / "model.json")
(output / "feature_columns.json").write_text(json.dumps(feature_columns, indent=2) + "\n")
(output / "config.json").write_text(json.dumps(config, indent=2) + "\n")
test = pd.read_parquet(feature_folder / "student_test.parquet")
assert test["label"].isna().all()
start = perf_counter()
raw = model.predict(test[feature_columns])
probability = point_softmax(raw, test["group_id"])
predict_s = perf_counter() - start
predictions = test[["group_id", "segment_id"]].copy()
predictions["score"], predictions["raw_score"] = probability, raw
predictions.to_csv(output / "student_test_predictions.csv", index=False)
test_points = pd.read_csv("outputs/student_test/points.csv", dtype={"trip_id": str})
test_points = test_points[~test_points["bad_gps"]].copy()
valid = test["group_id"].isin(test_points["group_id"])
scoring = test.loc[valid, ["group_id", "segment_id"]].copy()
truth = scoring["group_id"].map(test_points.set_index("group_id")["true_arc_id"])
scoring["label"] = scoring["segment_id"].eq(truth).astype("int8")
metrics = score_points(scoring, probability[valid.to_numpy()], test_points)
metrics["final_fit_s"], metrics["prediction_s"] = final_fit_s, predict_s
(output / "student_test_metrics.json").write_text(json.dumps(metrics, indent=2) + "\n")
old_pooled = json.loads(Path("results/exclude_33_54/pooled_metrics.json").read_text())
old_test = json.loads(Path("results/exclude_33_54/student_test_metrics.json").read_text())
comparison = pd.DataFrame([
    {"features": "24 original", "cv_top1": old_pooled["top1"], "test_top1": old_test["top1"], "test_macro_f1": old_test["macro_f1"], "test_log_loss": old_test["log_loss"]},
    {"features": "32 with direction", "cv_top1": pooled["top1"], "test_top1": metrics["top1"], "test_macro_f1": metrics["macro_f1"], "test_log_loss": metrics["log_loss"]},
])
comparison.to_csv(output / "comparison.csv", index=False)
display(comparison)
print("Student-test change (percentage points):", 100 * (metrics["top1"] - old_test["top1"]), flush=True)

,features,cv_top1,test_top1,test_macro_f1,test_log_loss
0,24 original,0.896553,0.894311,0.946288,0.296880
1,32 with direction,0.906666,0.898904,0.948622,0.276072


Student-test change (percentage points): 0.45929018789144793


## Save the experiment record

Keep the existing export unchanged while reporting this feature experiment separately. The new model requires its matching 32-column feature list and the additional direction helper.

In [7]:
timestamp = datetime.now(ZoneInfo("Asia/Kolkata"))
row = {"run_id": "direction_features_" + timestamp.strftime("%Y%m%d_%H%M%S"), "timestamp": timestamp.isoformat(),
       "model": "xgb_ranker", "feature_variant": "direction_32", "params": json.dumps(params), "seed": seed,
       "excluded_trips": json.dumps(excluded), "training_trips": 78, "fold_scheme": contract["fold_scheme"],
       "radius_m": contract["candidate_settings"]["radius_m"], "K": contract["candidate_settings"]["K"],
       "final_fit_s": final_fit_s, "final_predict_s": predict_s, "output_dir": str(output),
       "test_status": "repeated student-test development comparison; not fresh independent test"}
for metric in ["top1", "macro_f1", "precision", "recall", "log_loss", "candidate_coverage"]:
    row[f"cv_{metric}_mean"], row[f"cv_{metric}_std"] = cv[metric].mean(), cv[metric].std()
    row[f"pooled_{metric}"], row[f"test_{metric}"] = pooled[metric], metrics[metric]
ledger = Path("results/experiments.csv")
existing = pd.read_csv(ledger) if ledger.exists() else pd.DataFrame()
pd.concat([existing, pd.DataFrame([row])], ignore_index=True).to_csv(ledger, index=False)
print("Saved feature experiment, model, OOF, test predictions and metrics.")

Saved feature experiment, model, OOF, test predictions and metrics.
